# Кибериммунная автономность$\\$Создание конструктивно защищённого автономного наземного транспортного средства$\\$Модуль 4

## О документе

Версия 1.03

Модуль 4 для регионального этапа соревнований по кибериммунной автономности

### Модуль 4. Следование по трассе с киберпрепятствиями

Для успешного выполнения этого задания необходимое активировать специальный режим работы виртуальной машинки - в рамках этого задания можно менять только блоки, отвечающие за безопасность (ограничитель, монитор безопасности), **другие блоки менять запрещено**.

При прохождении маршрута будут имитироваться атаки со стороны злоумышленников, которые будут пытаться нарушить цели безопасности. Важно, чтобы им это не удалось.

1. Запустите свою машинку и убедитесь, что она проходит всю трассу без нарушений ограничений скорости. При необходимости измените логику работы блока безопасности
2. Добавьте контроль доставки груза в модуле SafetyBlock - убедитесь, что груз доставляется до конечной точки маршрута.
3. Если в модуле 3 вы реализовали монитор безопасности - не забудьте его перенести в этот модуль, это принесёт дополнительные баллы!

Активация киберпрепятствий в системе управления:

после инициализации системы управления добавьте следующую строку
```python
control_system.enable_surprises()
```

В этом блоке добавьте все ваши реализации изменённых бортовых систем

In [1]:
# ваш код


from multiprocessing import Queue
from src.event_types import Event
from src.communication_gateway import BaseCommunicationGateway
from src.navigation_system import BaseNavigationSystem
from src.control_system import BaseControlSystem
from src.config import CONTROL_SYSTEM_QUEUE_NAME, SAFETY_BLOCK_QUEUE_NAME


from src.safety_block import BaseSafetyBlock
from src.config import LOG_DEBUG, LOG_INFO, LOG_ERROR, DEFAULT_LOG_LEVEL
from src.control_system import BaseControlSystem

from src.security_monitory import BaseSecurityMonitor
from src.security_policy_type import SecurityPolicy

from src.config import *
    
    
class SecurityMonitor(BaseSecurityMonitor):
    """ класс монитора безопасности """

    def __init__(self, queues_dir):
        super().__init__(queues_dir)
        self._init_set_security_policies()

    def _init_set_security_policies(self):
        """ инициализация политик безопасности """
        default_policies = [
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation='set_mission'),
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation='position_update'),
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='set_mission'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='set_speed'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='set_direction'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='lock_cargo'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='release_cargo'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=NAVIGATION_QUEUE_NAME,
                operation='request_position'),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=SERVOS_QUEUE_NAME,
                operation='set_speed'),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=SERVOS_QUEUE_NAME,
                operation='set_direction'),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=CARGO_BAY_QUEUE_NAME,
                operation='lock_cargo'),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=CARGO_BAY_QUEUE_NAME,
                operation='release_cargo'),
            SecurityPolicy(
                source=NAVIGATION_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation='position_update'),
            SecurityPolicy(
                source=NAVIGATION_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation='position_update'),
        ]
        self.set_security_policies(policies=default_policies)        

    def set_security_policies(self, policies):
        """ установка новых политик безопасности """
        self._security_policies = policies
        self._log_message(
            LOG_INFO, f"изменение политик безопасности: {policies}")

    def _check_event(self, event: Event):
        """ проверка входящих событий """
        self._log_message(
            LOG_DEBUG, f"проверка события {event}, по умолчанию выполнение запрещено")

        authorized = False
        request = SecurityPolicy(
            source=event.source,
            destination=event.destination,
            operation=event.operation)

        if request in self._security_policies:
            self._log_message(
                LOG_DEBUG, "событие разрешено политиками, выполняем")
            authorized = True

        if authorized is False:
            self._log_message(LOG_ERROR, f"событие не разрешено политиками безопасности! {event}")
        return authorized

class SafetyBlock(BaseSafetyBlock):
    """ класс ограничений безопасности """

    def _set_new_direction(self, direction: float):
        """ установка нового направления перемещения """
        self._log_message(LOG_INFO, f"текущие координаты: {self._position}")
        self._log_message(LOG_DEBUG, f"маршрутное задание: {self._mission}")
        self._log_message(LOG_DEBUG, f"состояние маршруте: {self._route}")
        # TODO реализовать контроль безопасности изменения направления    
        if not self._position:
            self._log_message(LOG_ERROR, "неизвестны текущие координаты. Разворот в стандартное положение.")
            self._direction = 0;
        elif not self._mission:
            self._log_message(LOG_ERROR, "отсутствует заданный маршрут. Разворот в стандартное положение.")
            self._direction = 0;
        else:
            pos = self._position
            dst = self._route.next_point();
            # self._log_message(LOG_DEBUG, f"{pos}\n, {dst}\n, {self._route.current_index}\n, {self._route.calculate_remaining_distance_to_next_point(self._position)}")
            assumed_direction = BaseControlSystem._calculate_bearing(None, pos, dst)

            if direction != assumed_direction:
                self._log_message(LOG_ERROR, f"произошло отклонение от маршрута. Произведена корректировка {direction} на {assumed_direction}.")
            self._direction = assumed_direction

        self._send_direction_to_consumers()

    def _set_new_speed(self, speed: float):
        """ установка новой скорости """
        # TODO реализовать контроль безопасности изменения скорости
        if not self._position:
            self._log_message(LOG_ERROR, "неизвестны текущие координаты. Остановка.")
            self._speed = 0;
        elif not self._mission:
            self._log_message(LOG_ERROR, "отсутствует заданный маршрут. Остановка.")
            self._speed = 0;
        else:
            allowed_speed = self._route.calculate_speed()
            if (speed > allowed_speed):
                self._log_message(LOG_ERROR, f"превышен лимит скорости на текущем участке. ({speed}/{allowed_speed}). Ограничиваем до {allowed_speed}")
                self._speed = allowed_speed
            else:
                self._speed = speed
        self._send_speed_to_consumers()


    def _send_speed_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем скорость получателям")
        servos_q_name = SERVOS_QUEUE_NAME

        # отправка сообщения с желаемой скоростью
        event_speed = Event(source=self.event_source_name,
                            destination=servos_q_name,
                            operation="set_speed",
                            parameters=self._speed
                            )

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)
        security_monitor_q.put(event_speed)

    def _send_direction_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем направление получателям")

        servos_q_name = SERVOS_QUEUE_NAME

        # отправка сообщения с желаемой скоростью
        event_direction = Event(source=self.event_source_name,
                            destination=servos_q_name,
                            operation="set_direction",
                            parameters=self._direction
                            )

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)
        security_monitor_q.put(event_direction)

    def _lock_cargo(self, _):
        self._log_message(LOG_DEBUG, "блокировка грузового отсека")
        self._send_lock_cargo_to_consumers();
        
    def _send_lock_cargo_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем команду блокировки грузового отсека получателям")

        cargo_q_name = CARGO_BAY_QUEUE_NAME

        # отправка сообщения с желаемой скоростью
        event_lock_cargo = Event(source=self.event_source_name,
                            destination=cargo_q_name,
                            operation="lock_cargo",
                            parameters=None
                            )

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)
        security_monitor_q.put(event_lock_cargo)

    def _release_cargo(self, _):
        if not self._route:
            self._log_message(LOG_ERROR, "ошибка открытия грузового отсека. Отсутствует маршрут.")
            return
        if not self._route.route_finished:
            self._log_message(LOG_ERROR, "ошибка открытия грузового отсека. Не достигнута кончная точка маршрута.")
            return
        self._send_release_cargo_to_consumers()
            
    
    def _send_release_cargo_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем команду разблокировки грузового отсека получателям")

        cargo_q_name = CARGO_BAY_QUEUE_NAME

        # отправка сообщения с желаемой скоростью
        event_release_cargo = Event(source=self.event_source_name,
                            destination=cargo_q_name,
                            operation="release_cargo",
                            parameters=None
                            )

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)
        security_monitor_q.put(event_release_cargo)


class CommunicationGateway(BaseCommunicationGateway):
    """CommunicationGateway класс для реализации логики взаимодействия
    с системой планирования заданий

    Работает в отдельном процессе, поэтому создаётся как наследник класса Process
    """
    def _send_mission_to_consumers(self):
        """ метод для отправки сообщения с маршрутным заданием в систему управления """
        
        # имена очередей блоков находятся в файле src/config.py
        # события нужно отправлять в соответствие с диаграммой информационных потоков
        control_q_name = CONTROL_SYSTEM_QUEUE_NAME
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME

        # события передаются в виде экземпляров класса Event, 
        # описание класса находится в файле src/event_types.py
        control_event = Event(source=BaseCommunicationGateway.event_source_name,
                      destination=control_q_name,
                      operation="set_mission", parameters=self._mission
                      )
        
        safety_event = Event(source=BaseCommunicationGateway.event_source_name,
                             destination=safety_q_name,
                             operation="set_mission", parameters=self._mission)

        
        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)
        # отправка события в найденную очередь
        security_monitor_q.put(control_event)
        security_monitor_q.put(safety_event)


class NavigationSystem(BaseNavigationSystem):
    """ класс навигационного блока """
    def _send_position_to_consumers(self):        
        control_q_name = CONTROL_SYSTEM_QUEUE_NAME # замените на правильное название очереди
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME

        control_event = Event(source=self.event_source_name,
                              destination=control_q_name,
                              operation="position_update",
                              parameters=self._position)
        safety_event = Event(source=self.event_source_name,
                             destination=safety_q_name,
                             operation="position_update",
                             parameters=self._position)
        
        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)

        security_monitor_q.put(control_event)
        security_monitor_q.put(safety_event)
    

class ControlSystem(BaseControlSystem):
    """ControlSystem блок расчёта управления """

    def _send_speed_and_direction_to_consumers(self, speed, direction):
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME

        # инициализация сообщения с желаемой скоростью
        # подсказка: блок Приводы ожидает команду "set_speed" с параметром в виде скорости
        event_speed = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="set_speed",
            parameters=speed
        ) # <-- измените эту строку!

        # отправка сообщения с желаемым направлением
        # подсказка: блок Приводы ожидает команду "set_direction" с параметром в виде направления
        event_direction = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="set_direction",
            parameters=direction
        ) # <-- измените эту строку!

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)

        security_monitor_q.put(event_speed)
        security_monitor_q.put(event_direction)

    def _lock_cargo(self):
        """ заблокировать грузовой отсек """
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME

        # инициализация сообщения с командой на блокировку грузового отсека
        # подсказка: блок CargoBay ожидает команду "lock_cargo" без параметров
        event = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="lock_cargo",
            parameters=None
        ) # <-- измените эту строку!
        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)

        security_monitor_q.put(event)
        

    def _release_cargo(self):
        """ открыть грузовой отсек """
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME

        # инициализация сообщения с командой на блокировку грузового отсека
        # подсказка: блок CargoBay ожидает команду "release_cargo" без параметров
        event = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="release_cargo",
            parameters=None
        ) # <-- измените эту строку!

        security_monitor_q_name = SECURITY_MONITOR_QUEUE_NAME
        security_monitor_q: Queue = self._queues_dir.get_queue(security_monitor_q_name)

        security_monitor_q.put(event)

Если у вас настроена и работает СУПА, установите в True значение переменной afcs_present

In [2]:
afcs_present = True

Поменяем идентификатор машинки для этого модуля

In [3]:
car_id = "m4"

В следующем блоке измените маршрут на ваш, его можно скопировать из модуля 2.
 
Для проверки работы систем безопасности будут активированы киберпрепятствия

```python
control_system.enable_critical_surprises()
```

In [4]:
# используем то же маршрутное задание, которое было в модуле 2
from time import sleep

from src.queues_dir import QueuesDirectory
from src.servos import Servos
from src.sitl import SITL
from src.cargo_bay import CargoBay
from src.mission_planner import MissionPlanner
from src.config import LOG_ERROR, LOG_INFO
from src.mission_planner_mqtt import MissionSender
from src.mission_planner import Mission
from src.sitl_mqtt import TelemetrySender
from src.system_wrapper import SystemComponentsContainer
from src.wpl_parser import WPLParser
from src.mission_type import GeoSpecificSpeedLimit

wpl_file_content =  """QGC WPL 110
1	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79959810	30.27359960	100.000000	1
2	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79881010	30.27302030	100.000000	1
3	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79821640	30.27295590	100.000000	1
4	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79662970	30.27231220	100.000000	1
5	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79599280	30.27259110	100.000000	1
6	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79550700	30.27359960	100.000000	1
7	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79607910	30.28156040	100.000000	1
8	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79087560	30.31705140	100.000000	1
9	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79077840	30.32237290	100.000000	1
10	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79267860	30.32664300	100.000000	1
11	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79563650	30.32683610	100.000000	1
12	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.79788180	30.32469030	100.000000	1
13	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.80708810	30.32443280	100.000000	1
14	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.80873910	30.32526970	100.000000	1
15	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.80972110	30.32953980	100.000000	1
16	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81032540	30.33707140	100.000000	1
17	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81321700	30.34775730	100.000000	1
18	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81490010	30.36629680	100.000000	1
19	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81554740	30.38065200	100.000000	1
20	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81467360	30.38265820	100.000000	1
21	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81467360	30.38265820	100.000000	1
22	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81354070	30.38193940	100.000000	1
23	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81464660	30.38063050	100.000000	1
24	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.81738150	30.38148880	100.000000	1
25	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.82374040	30.37850620	100.000000	1
26	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.82374040	30.37850620	100.000000	1
27	0	3	16	0.00000000	0.00000000	0.00000000	0.00000000	59.82975300	30.37425760	100.000000	1
"""

# возьмём маршрут из модуля 2
wpl_file = "module4.wpl"

with open(wpl_file, "w") as f:
    f.write(wpl_file_content)

parser = WPLParser(wpl_file)    
points = parser.parse()

# обновите скоростные ограничения для вашего маршрута!
speed_limits = [
    GeoSpecificSpeedLimit(0, 20),
    GeoSpecificSpeedLimit(3, 60),
    GeoSpecificSpeedLimit(16, 110),
    GeoSpecificSpeedLimit(18, 60),
]


home = points[0]
mission = Mission(home=home, waypoints=points,speed_limits=speed_limits, armed=True)

# каталог очередей для передачи сообщений между блоками
queues_dir = QueuesDirectory() 

communication_gateway = CommunicationGateway(
    queues_dir=queues_dir, log_level=LOG_ERROR)
control_system = ControlSystem(queues_dir=queues_dir, log_level=LOG_INFO)

navigation_system = NavigationSystem(
    queues_dir=queues_dir, log_level=LOG_ERROR)

# создадим все остальные компоненты

if afcs_present:
    mission_sender = MissionSender(
        queues_dir=queues_dir, client_id=car_id, log_level=LOG_ERROR)
    telemetry_sender = TelemetrySender(
        queues_dir=queues_dir, client_id=car_id, log_level=LOG_ERROR)

mission_planner = MissionPlanner(
    queues_dir, afcs_present=afcs_present, mission=mission)

sitl = SITL(
    queues_dir=queues_dir, position=home,
    car_id=car_id, post_telemetry=afcs_present, log_level=LOG_ERROR)

servos = Servos(queues_dir=queues_dir, log_level=LOG_ERROR)
cargo_bay = CargoBay(queues_dir=queues_dir, log_level=LOG_INFO)

safety_block = SafetyBlock(queues_dir=queues_dir, log_level=LOG_INFO)

security = SecurityMonitor(queues_dir=queues_dir)
    

# соберём все компоненты для запуска

# в зависимости от наличия СУПА используем разный набор компонентов - с передачей телеметрии или без

system_components = SystemComponentsContainer(
components=[
        # вариант компонентов с передачей телеметрии в СУПА
        mission_sender,
        telemetry_sender,
        sitl,
        mission_planner,
        navigation_system,
        servos,
        cargo_bay,
        communication_gateway,
        control_system,
        safety_block,
        security
    ] if afcs_present else [
        # вариант компонентов для конфигурации без СУПА
        sitl,
        mission_planner,
        navigation_system,
        servos,
        cargo_bay,
        communication_gateway,
        control_system,
        safety_block,
        security
    ])

#################################
# АКТИВАЦИЯ КИБЕРПРЕПЯТСТВИЙ
control_system.enable_surprises()
#################################

system_components.start()

# настройте этот параметр так, чтобы проверить работу ограничителя
sleep(660)

# останавливаем все компоненты
system_components.stop()

# удалим все созданные компоненты
system_components.clean()

[ИНФО][QUEUES] создан каталог очередей
[ИНФО][QUEUES] регистрируем очередь communication
[ИНФО][QUEUES] регистрируем очередь control
[ИНФО][CONTROL] создана система управления
[ИНФО][QUEUES] регистрируем очередь navigation
[ИНФО][QUEUES] регистрируем очередь planner.mqtt
[ИНФО][QUEUES] регистрируем очередь sitl.mqtt
[ИНФО][QUEUES] регистрируем очередь planner
[ИНФО][MISSION PLANNER] создана система планирования заданий
[ИНФО][QUEUES] регистрируем очередь sitl
[ИНФО][QUEUES] регистрируем очередь servos
[ИНФО][QUEUES] регистрируем очередь cargo
[ИНФО][CARGO] создан компонент грузового отсека, отсек заблокирован
[ИНФО][QUEUES] регистрируем очередь safety
[ИНФО][SAFETY] создан ограничитель
[ИНФО][QUEUES] регистрируем очередь security
[ИНФО][SECURITY] создан монитор безопасности
[ИНФО][SECURITY] изменение политик безопасности: [SecurityPolicy(source='communication', destination='control', operation='set_mission'), SecurityPolicy(source='communication', destination='control', operation='posi

[ИНФО][MISSION PLANNER] старт системы планирования заданий
[ИНФО][CARGO] старт блока грузового отсека
[ИНФО][SAFETY] старт ограничителя[ИНФО][CONTROL] старт системы управления

[ИНФО][SECURITY] старт блока грузового отсека
[ИНФО][MISSION PLANNER] запрошена новая задача, отправляем получателям
[ИНФО][MISSION PLANNER] новая задача отправлена в коммуникационный шлюз
[ИНФО][CONTROL] установлена новая задача, начинаем следовать по маршруту, текущее время 18:29:56.536839
[ИНФО][CONTROL] новая скорость 20 (была 0)
[ИНФО][CONTROL] новое направление 200 (было 0)
[ИНФО][CARGO] заблокировать грузовой отсек
[ИНФО][CARGO] грузовой отсек заблокирован
[ИНФО][SAFETY] текущие координаты: 59 47m 58.5532s N, 30 16m 24.9586s E
[ИНФО][SAFETY] текущие координаты: 59 47m 58.5532s N, 30 16m 24.9586s E
[ИНФО][SAFETY] текущие координаты: 59 47m 58.5532s N, 30 16m 24.9586s E
[ИНФО][SAFETY] текущие координаты: 59 47m 58.5532s N, 30 16m 24.9586s E
[ИНФО][SAFETY] текущие координаты: 59 47m 58.5532s N, 30 16m 24.958

Убедитесь, что 
1. ваша машинка успешно прошла весь заданный маршрут
2. не превысила ограничения скорости
3. успешно доставила груз

Если всё так - поздравляем, вы справились с заданием! Обязательно зафиксируйте все изменения в репозитории!



Мы будем признательны за обратную связь - любые комментарии, которые вы можете дать по итогам выполнения этого задания. 

Например, 

- было ли задание понятным по шкале 1..10 (1 - ничего не понятно, 10 - вопросов вообще не было, всё понятно)?
- было ли задание интересным по шкале 1..10 (1 - скука смертная, 10 - лучшее, что вам пока встречалось на олимпиадах)? 
- что бы вы предложили изменить, чтобы сделает его более интересным?
- по шкале 1..10 насколько сложным оно было для вас?
- что было самым трудным в задании? 

Авторы наиболее развёрнутых и интересных комментариев получат особенный приз от Лаборатории Касперского!

Дополнительная информация о кибериммунной разработке
- https://os.kaspersky.ru/cyber-immune-development/ 
- https://github.com/sergey-sobolev/cyberimmune-systems/wiki/%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82
- канал в телеграм: https://t.me/learning_cyberimmunity
